In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


CPU threads set to: 32
CUDA device: NVIDIA GeForce RTX 3090


# Kvasir-VQA x1 — Text-only BERT classifier (top-K answers)

Fine-tune a BERT-style encoder on question text for top-K answer classification, using the same splits and label mapping as the TF-IDF baseline. Outputs mirror existing x1 conventions.


In [2]:
from pathlib import Path
import json
import random
import math
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    top_k_accuracy_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)
from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [3]:
# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "01_text_only" / "out" / "02_bert"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "bert-base-uncased"  # stronger baseline than distilbert
TOP_K = None  # use full answer space
MAX_LENGTH = 128
BASE_BATCH_SIZE = 16
BASE_EFF_BATCH = 32
BATCH_SIZE = BASE_BATCH_SIZE
GRAD_ACCUM_STEPS = 2  # will be auto-tuned
EPOCHS = 6
LR = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
PATIENCE = 2
GRAD_CLIP = 1.0
VAL_SPLIT = 0.05  # create a val split from train if none exists
CLASS_WEIGHT_POWER = 0.5  # soften inverse-frequency weights
LABEL_SMOOTHING = 0.1
MAX_CONFUSION_CLASSES = 2000  # skip huge confusion matrices
MAX_PER_CLASS = 2000  # skip per-class report for huge label spaces
MAX_PROB_STORE_CLASSES = 5000  # skip saving full probs when too many classes
NUM_WORKERS = min(8, CPU_THREADS)

# Auto-tune batch size for GPU memory (keeps effective batch ~BASE_EFF_BATCH)
if torch.cuda.is_available():
    mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if mem_gb >= 20:
        BATCH_SIZE = 16
    elif mem_gb >= 12:
        BATCH_SIZE = 8
    elif mem_gb >= 8:
        BATCH_SIZE = 4
    else:
        BATCH_SIZE = 2
else:
    BATCH_SIZE = 2
GRAD_ACCUM_STEPS = max(1, BASE_EFF_BATCH // BATCH_SIZE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUM_STEPS)


Data root: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/01_text_only/out/02_bert
Device: cuda
Effective batch size: 32


In [4]:
# Load metadata and splits
meta = pd.read_csv(META_CSV)

meta["question_norm"] = meta["question"].fillna("").astype(str).str.lower().str.strip()
meta["answer_norm"] = meta["answer"].fillna("").astype(str).str.lower().str.strip()

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 143594, 'val': 0, 'test': 15955}


In [5]:
# Answer space selection
if TOP_K is None:
    # Use full train answer space (filter val/test to seen answers)
    train_k = train_df[train_df["answer_norm"].notna()].reset_index(drop=True)
    TOP_K_ANSWERS = sorted(train_k["answer_norm"].unique().tolist())
    allowed = set(TOP_K_ANSWERS)
    val_k = val_df[val_df["answer_norm"].isin(allowed)].reset_index(drop=True)
    test_k = test_df[test_df["answer_norm"].isin(allowed)].reset_index(drop=True)
    dropped_val = len(val_df) - len(val_k)
    dropped_test = len(test_df) - len(test_k)
    print("Using full train answer space:", len(TOP_K_ANSWERS))
    print({"train": len(train_k), "val": len(val_k), "test": len(test_k)})
    if dropped_val or dropped_test:
        print("Dropped rows with unseen answers:", {"val": dropped_val, "test": dropped_test})
else:
    answer_counts = train_df["answer_norm"].value_counts()
    TOP_K_ANSWERS = answer_counts.head(TOP_K).index.tolist()
    train_k = train_df[train_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
    val_k = val_df[val_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
    test_k = test_df[test_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
    print("Top-K answers:", len(TOP_K_ANSWERS))
    print({"train": len(train_k), "val": len(val_k), "test": len(test_k)})


Using full train answer space: 72641
{'train': 143594, 'val': 0, 'test': 9148}
Dropped rows with unseen answers: {'val': 0, 'test': 6807}


In [6]:
# Create validation split if missing (from train_k)
if len(val_k) == 0 and len(train_k):
    try:
        train_k, val_k = train_test_split(
            train_k,
            test_size=VAL_SPLIT,
            random_state=SEED,
            stratify=train_k["answer_norm"],
        )
    except ValueError:
        train_k, val_k = train_test_split(
            train_k,
            test_size=VAL_SPLIT,
            random_state=SEED,
        )
    train_k = train_k.reset_index(drop=True)
    val_k = val_k.reset_index(drop=True)
    print("Created val split from train_k:", {"train": len(train_k), "val": len(val_k)})


Created val split from train_k: {'train': 136414, 'val': 7180}


In [7]:
# Label mapping
classes = sorted(TOP_K_ANSWERS)
class_to_idx = {c: i for i, c in enumerate(classes)}

def encode_labels(df):
    labels = df["answer_norm"].map(class_to_idx)
    if labels.isna().any():
        bad = int(labels.isna().sum())
        raise RuntimeError(f"Found {bad} labels not in class map. Check answer space filtering.")
    return labels.astype(int).values

train_labels = encode_labels(train_k)
val_labels = encode_labels(val_k) if len(val_k) else None
test_labels = encode_labels(test_k) if len(test_k) else None

print("#classes:", len(classes))
print("train_labels:", train_labels.dtype, int(train_labels.min()), int(train_labels.max()))
if val_labels is not None and len(val_labels):
    print("val_labels:", val_labels.dtype, int(val_labels.min()), int(val_labels.max()))
if test_labels is not None and len(test_labels):
    print("test_labels:", test_labels.dtype, int(test_labels.min()), int(test_labels.max()))


#classes: 72641
train_labels: int64 0 72640
val_labels: int64 7 72599
test_labels: int64 18 72583


In [8]:
# Tokenizer and datasets
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class QAQuizDataset(Dataset):
    def __init__(self, df, labels=None):
        self.texts = df["question_norm"].tolist()
        self.labels = None
        if labels is not None:
            try:
                self.labels = np.asarray(labels, dtype=np.int64)
            except Exception as e:
                raise RuntimeError("Labels could not be converted to int64. Re-run label mapping.") from e
            if not np.isfinite(self.labels).all():
                raise RuntimeError("Labels contain non-finite values. Re-run answer space selection.")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = QAQuizDataset(train_k, train_labels)
val_ds = QAQuizDataset(val_k, val_labels) if val_labels is not None else None
test_ds = QAQuizDataset(test_k, test_labels) if test_labels is not None else None


In [9]:
# DataLoaders
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
) if val_ds else None
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
) if test_ds else None


In [10]:
# Class weights (optional)
class_weights = None
counts = Counter(train_labels)
if len(counts) and len(classes):
    weights = np.zeros(len(classes), dtype=np.float32)
    for c, idx in class_to_idx.items():
        denom = len(classes) * counts.get(idx, 0)
        if denom == 0:
            weights[idx] = 0.0
            continue
        # soften inverse-frequency to reduce over-penalizing rare classes
        weights[idx] = (len(train_labels) / denom) ** CLASS_WEIGHT_POWER
    class_weights = torch.tensor(weights, dtype=torch.float32)  # keep on CPU first
    try:
        class_weights = class_weights.to(DEVICE)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            print("Class weights remain on CPU because GPU is full; will drop weighting.")
            class_weights = None
        else:
            raise
else:
    print("Skipping class weights: empty classes or labels.")
print("Using class weights:", class_weights is not None)


Using class weights: True


In [11]:
# Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(classes),
)
model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = EPOCHS * steps_per_epoch
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(WARMUP_RATIO * total_steps), num_training_steps=total_steps
)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


2026-01-29 13:02:52.254751: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-29 13:02:52.254788: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-29 13:02:52.255849: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-29 13:02:52.261182: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-29 13:02:53.008859: W tensorflow/compiler/tf2

In [12]:
# Training & evaluation helpers

def _compute_metrics(y_true, y_pred, probs, labels, top3=None, top5=None):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

    if top3 is not None:
        metrics["top_3_accuracy"] = float(top3)
    if top5 is not None:
        metrics["top_5_accuracy"] = float(top5)

    per_class = None
    if len(labels) <= MAX_PER_CLASS:
        pr, rc, f1, support = precision_recall_fscore_support(
            y_true, y_pred, labels=list(range(len(labels))), zero_division=0
        )
        per_class = {}
        for i, name in enumerate(labels):
            per_class[str(name)] = {
                "precision": float(pr[i]),
                "recall": float(rc[i]),
                "f1": float(f1[i]),
                "support": int(support[i]),
            }
        metrics["per_class"] = per_class
    else:
        metrics["per_class_skipped"] = f"too_many_classes({len(labels)})"

    if probs is not None:
        n_classes = probs.shape[1]
        if n_classes == 2:
            pos = probs[:, 1]
            try:
                metrics["roc_auc"] = float(roc_auc_score(y_true, pos))
                metrics["pr_auc"] = float(average_precision_score(y_true, pos))
                metrics["brier"] = float(brier_score_loss(y_true, pos))
            except Exception:
                pass
    return metrics


def run_eval(loader):
    model.eval()
    all_labels, all_preds = [], []
    all_probs = []
    store_probs = len(classes) <= MAX_PROB_STORE_CLASSES
    top3_correct = 0
    top5_correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**{k: v for k, v in batch.items() if k != "labels"})
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)
            labels = batch["labels"]

            # Top-k accuracy streaming
            kmax = 5
            topk = torch.topk(logits, k=min(kmax, logits.size(1)), dim=1).indices
            if topk.size(1) >= 3:
                top3_correct += (topk[:, :3] == labels.unsqueeze(1)).any(dim=1).sum().item()
            if topk.size(1) >= 5:
                top5_correct += (topk[:, :5] == labels.unsqueeze(1)).any(dim=1).sum().item()
            total += labels.size(0)

            if store_probs:
                probs = torch.softmax(logits, dim=1)
                all_probs.append(probs.cpu())
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    y_true = torch.cat(all_labels).numpy()
    y_pred = torch.cat(all_preds).numpy()
    probs = torch.cat(all_probs).numpy() if store_probs and len(all_probs) else None

    top3 = top3_correct / total if total else None
    top5 = top5_correct / total if total else None

    metrics = _compute_metrics(y_true, y_pred, probs, classes, top3=top3, top5=top5)

    report = None
    if len(classes) <= MAX_PER_CLASS:
        report = classification_report(
            y_true, y_pred,
            labels=list(range(len(classes))),
            target_names=classes,
            output_dict=True,
            zero_division=0,
        )
    y_pred_lbl = [classes[i] for i in y_pred]
    return metrics, report, y_pred_lbl, probs, y_true, y_pred


def save_metrics(prefix, split_name, metrics, report, extra=None):
    payload = {"metrics": metrics, "report": report}
    if extra is not None:
        payload["extra"] = extra
    with open(OUT_DIR / f"{prefix}_metrics_{split_name}.json", "w") as f:
        json.dump(payload, f, indent=2)


def save_predictions(prefix, split_name, df, preds):
    out_df = df[["question", "answer", "answer_norm"]].copy()
    out_df["pred"] = preds
    out_df.to_csv(OUT_DIR / f"{prefix}_pred_{split_name}.csv", index=False)


def save_probs(prefix, split_name, probs, ids):
    if probs is None:
        return
    np.savez(OUT_DIR / f"{prefix}_probs_{split_name}.npz", probs=probs, classes=np.array(classes), ids=np.array(ids))


def save_confusion(prefix, split_name, y_true, y_pred):
    if len(classes) > MAX_CONFUSION_CLASSES:
        return
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)
    cm_df.to_csv(OUT_DIR / f"{prefix}_confusion_{split_name}.csv")


In [13]:
# Training loop with early stopping on val macro-F1
best_state = None
best_val_f1 = -1.0
patience = PATIENCE
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(**{k: v for k, v in batch.items() if k != "labels"})
            loss = criterion(outputs.logits, batch["labels"])
        epoch_loss += loss.item() * batch["labels"].size(0)
        loss = loss / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    epoch_loss /= len(train_loader.dataset)

    log = {"epoch": epoch, "train_loss": epoch_loss}

    if val_loader is not None:
        val_metrics, _, _, _, _, _ = run_eval(val_loader)
        log.update(val_metrics)
        val_f1 = val_metrics["macro_f1"]
        if val_f1 > best_val_f1 + 1e-4:
            best_val_f1 = val_f1
            best_state = model.state_dict()
            patience = PATIENCE
        else:
            patience -= 1
            if patience == 0:
                print("Early stopping")
                history.append(log)
                break
    history.append(log)
    print(log)

if best_state is not None:
    model.load_state_dict(best_state)


/tmp/ipykernel_759380/1189023110.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


{'epoch': 1, 'train_loss': 11.767826303965304, 'accuracy': 0.011420612813370474, 'macro_f1': 1.2572231418048615e-05, 'weighted_f1': 0.000701825991921772, 'balanced_accuracy': 0.0003940110323089047, 'top_3_accuracy': 0.022701949860724234, 'top_5_accuracy': 0.033008356545961, 'per_class_skipped': 'too_many_classes(72641)'}


/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  ys_types = set(type_of_target(x) for x in ys)
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home

{'epoch': 2, 'train_loss': 11.636237170952235, 'accuracy': 0.027715877437325905, 'macro_f1': 0.0001503456325297436, 'weighted_f1': 0.004824189405554419, 'balanced_accuracy': 0.0010414090585306908, 'top_3_accuracy': 0.04902506963788301, 'top_5_accuracy': 0.06991643454038997, 'per_class_skipped': 'too_many_classes(72641)'}


/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  ys_types = set(type_of_target(x) for x in ys)
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home

...

/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  ys_types = set(type_of_target(x) for x in ys)
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home

{'epoch': 5, 'train_loss': 11.305639229705585, 'accuracy': 0.08955431754874651, 'macro_f1': 0.0014654146881152039, 'weighted_f1': 0.036401433895573126, 'balanced_accuracy': 0.004940201663533901, 'top_3_accuracy': 0.16142061281337047, 'top_5_accuracy': 0.19610027855153203, 'per_class_skipped': 'too_many_classes(72641)'}


/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  ys_types = set(type_of_target(x) for x in ys)
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home

{'epoch': 6, 'train_loss': 11.22179394849683, 'accuracy': 0.0894150417827298, 'macro_f1': 0.001436004935521848, 'weighted_f1': 0.036347775568156444, 'balanced_accuracy': 0.004888029916997504, 'top_3_accuracy': 0.16559888579387186, 'top_5_accuracy': 0.20445682451253483, 'per_class_skipped': 'too_many_classes(72641)'}


/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  ys_types = set(type_of_target(x) for x in ys)
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home

In [14]:
# Evaluate and save
train_metrics, train_report, train_preds, train_probs, train_y_true, train_y_pred = run_eval(train_loader)
save_metrics("bert_text", "train", train_metrics, train_report)
save_predictions("bert_text", "train", train_k, train_preds)
save_probs("bert_text", "train", train_probs, train_k["img_id"].values if "img_id" in train_k.columns else np.arange(len(train_k)))
save_confusion("bert_text", "train", train_y_true, train_y_pred)

val_metrics = val_report = val_preds = val_probs = None
if val_loader is not None:
    val_metrics, val_report, val_preds, val_probs, val_y_true, val_y_pred = run_eval(val_loader)
    save_metrics("bert_text", "val", val_metrics, val_report)
    save_predictions("bert_text", "val", val_k, val_preds)
    save_probs("bert_text", "val", val_probs, val_k["img_id"].values if "img_id" in val_k.columns else np.arange(len(val_k)))
    save_confusion("bert_text", "val", val_y_true, val_y_pred)

if test_loader is not None:
    test_metrics, test_report, test_preds, test_probs, test_y_true, test_y_pred = run_eval(test_loader)
    save_metrics("bert_text", "test", test_metrics, test_report)
    save_predictions("bert_text", "test", test_k, test_preds)
    save_probs("bert_text", "test", test_probs, test_k["img_id"].values if "img_id" in test_k.columns else np.arange(len(test_k)))
    save_confusion("bert_text", "test", test_y_true, test_y_pred)
else:
    test_metrics = None

pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)

# Save model & tokenizer
(model_path := OUT_DIR / "bert_text_model").mkdir(parents=True, exist_ok=True)
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print("Saved model to", model_path)


/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  ys_types = set(type_of_target(x) for x in ys)
/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home

Saved model to /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/01_text_only/out/02_bert/bert_text_model


In [15]:
# Summary table
summary_rows = []
for name, mets in [("train", train_metrics), ("val", val_metrics), ("test", test_metrics)]:
    if mets:
        summary_rows.append({"split": name, "accuracy": mets["accuracy"], "macro_f1": mets["macro_f1"]})

if summary_rows:
    display(pd.DataFrame(summary_rows).set_index("split").round(4))
else:
    print("No metrics collected.")


,accuracy,macro_f1
split,,
train,0.0918,0.0001
val,0.0894,0.0014
test,0.1589,0.0023
